<a href="https://colab.research.google.com/github/aquilino/A-Simple-printf/blob/master/IA_esp32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
# Datos sintéticos: [feature1, feature2, feature3], etiquetas: 0=feliz, 1=triste, 2=enfadado
X_train = np.array([
    # Feliz (movimiento alto, frecuencia media, variabilidad baja)
    [0.8, 0.5, 0.2], [0.7, 0.6, 0.1], [0.9, 0.4, 0.3],

    # Triste (movimiento bajo, frecuencia baja, variabilidad baja)
    [0.1, 0.2, 0.1], [0.2, 0.1, 0.0], [0.3, 0.3, 0.2],

    # Enfadado (movimiento medio, frecuencia alta, variabilidad alta)
    [0.5, 0.9, 0.8], [0.4, 0.8, 0.7], [0.6, 0.7, 0.9]
], dtype=np.float32)

y_train = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2], dtype=np.int32)

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(3,)),  # Capa oculta
    tf.keras.layers.Dense(3, activation='softmax')  # Salida: 3 emociones
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("comenzando entrenamiento....")
model.fit(X_train, y_train, epochs=100)
print("Modelo entrenado...")

In [ ]:
import tensorflow as tf
import numpy as np

# Assume 'model' is your trained model from the previous code

# Sample input data representing a happy emotion
sample_input = np.array([[0.1, 0.8, 0.1]], dtype=np.float32)

# Make the prediction
prediction = model.predict(sample_input)

# Get the predicted class (emotion)
predicted_class = np.argmax(prediction)

# Interpret the prediction
emotion_labels = {0: "feliz", 1: "triste", 2: "enfadado"}
predicted_emotion = emotion_labels[predicted_class]

# Print the results
print("Sample Input:", sample_input)
print("Prediction (Raw Output):", prediction)
print("Predicted Class:", predicted_class)
print("Predicted Emotion:", predicted_emotion)

# Convertir a TFLite con cuantización
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

# Guardar el modelo quantizado
with open('emotion_model_quant.tflite', 'wb') as f:
    f.write(tflite_quant_model)

# Nueva sección

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Probar con un dato de ejemplo
input_data = np.array([[0.5, 0.9, 0.8]], dtype=np.float32)  # Enfadado
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])
print("Predicción:", np.argmax(output_data))  # Debería imprimir "2"

In [ ]:
# @title Texto de título predeterminado
# Convertir a TFLite sin cuantización (para simplicidad)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Guardar el modelo
with open('emotion_model.tflite', 'wb') as f:
    f.write(tflite_model)

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

with open('emotion_model_quant.tflite', 'wb') as f:
    f.write(tflite_quant_model)

In [ ]:
!xxd -i emotion_model_quant.tflite > model.h